In [2]:
import requests
from bs4 import BeautifulSoup

import regex as re
from datetime import datetime
import time
import random

from urllib.parse import urljoin

In [3]:
# Extraer todas las URLs de noticias
# ESTA FUNCIÓN DEBE REFINARSE PARA PODER SCRAPEAR CORRECTAMENTE LA SECCIÓN

# Resulta que en un primer vistazo la página solo ofrece unas pocas noticias, pero si se pulsa un botón se 
# pueden ver más, eso complica bastante la función

def extraer_urls(periodico, seccion, extension, limit=7, limite_peticiones=10, limite_urls=50):
    base = f"https://{periodico}"
    urls = []
    start = 0
    contador = 0

    headers = {"User-Agent": "Mozilla/5.0"}

    while (contador < limite_peticiones) and (len(urls) <= limite_urls):
        url = f"{base}/{seccion}?limit={limit}&start={start}&tmpl=component"
        resp = requests.get(url, headers=headers)

        if resp.status_code != 200:
            break

        soup = BeautifulSoup(resp.text, "html.parser")
        enlaces = soup.find_all("a", href=True)

        nuevos = []
        for a in enlaces:
            href = a["href"]
            if href.startswith("/"):
                href = urljoin(base, href)

            if href.startswith(f"{base}/{extension}/"):
                nuevos.append(href)

        if not nuevos:
            break

        for u in nuevos:
            if u not in urls:
                urls.append(u)

        start += limit
        contador += 1

    return urls


In [4]:
def extraer_noticia(url, nombre_seccion, nombre_periodico):
    
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(url, headers=headers)
    soup = BeautifulSoup(resp.text, "html.parser")

    # Título
    h1 = soup.find("h1", class_="article-title")
    if h1:
        a = h1.find("a")
        titulo = a.get_text(strip=True) if a else ""
    else:
        titulo = ""

    # Subtítulo
    subtitulo = soup.find("h2")
    subtitulo = subtitulo.get_text(strip=True) if subtitulo else ""

    # Texto del artículo
    cuerpo = soup.find("section", class_="article-content clearfix")

    if cuerpo:
        parrafos = cuerpo.find_all("p")
        texto = "\n".join(p.get_text(" ", strip=True).replace("  ", " ") for p in parrafos)
    else:
        texto = ""
    
    fecha_actual = datetime.now().strftime("%d-%m-%Y")

    noticia = {
            "Link": url,
            "Periódico": nombre_periodico,
            "Fecha": fecha_actual,
            "Título": titulo,
            "Subtítulo": subtitulo if subtitulo else None,
            "Categoría": nombre_seccion,
            "Contenido": texto
        }
    return noticia


In [5]:
url = "https://www.mediterraneodigital.com/internacional/mali-sahel-yihadismo-amenaza-espana-analisis-seguridad"

noticia = extraer_noticia(url, "sucesos-espana", "mediterraneodigital")
print(" ========== ========== ========= ======== =========")
print("------------ ###### TITULO ###### ---------")
print(noticia["Título"])
print("/n")
print("------------ ###### SUBTITULO ###### ---------")
print(noticia["Subtítulo"])
print("/n")
print("------------ ###### CONTENIDO ###### ---------")
print(noticia["Contenido"])
print("/n")

 ========== ========== ========= ======== =========
------------ ###### TITULO ###### ---------
Mali se desmorona y el Sahel amenaza a Europa: el riesgo real que España sigue ignorando
/n
------------ ###### SUBTITULO ###### ---------
None
/n
------------ ###### CONTENIDO ###### ---------
Hay una pregunta que los analistas de seguridad llevamos tiempo haciéndonos en voz baja y que ya es hora de formular en público: ¿cuánto tiempo más puede Occidente permitirse mirar al Sahel como si fuera un problema ajeno?
Mali se desmorona. No es una metáfora ni una exageración periodística. Es la evaluación que manejan los servicios de información españoles, que monitorizan con creciente preocupación el colapso progresivo del Gobierno de Bamako y el agotamiento de unas fuerzas armadas malienses incapaces de contener el empuje yihadista. La hipótesis de que los grupos armados alcancen la capital y proclamen alguna forma de Estado islámico ha dejado de ser un escenario de planificación para convertirs

In [6]:
def scrappeo_seccion(periodico, nombre_periodico, 
                     seccion, nombre_seccion,
                     extension,
                     limite_peticiones, limite_urls):
    
    urls = extraer_urls(periodico=periodico, seccion=seccion, extension=extension, 
                        limit=7, limite_peticiones=limite_peticiones, limite_urls=limite_urls)
    noticias = []
    i=1
    for url in urls:
        noticias.append(extraer_noticia(url, nombre_periodico=nombre_periodico, nombre_seccion=nombre_seccion))
        time.sleep(2)
        print(f"Noticia {i} de {len(urls)}")
        i+=1
    return noticias

url_base = "https:/"
periodico = "www.eldiario.es"
seccion = "internacional" 

# noticias = scrappeo_seccion(periodico="www.eldiario.es", seccion="internacional")

In [7]:
'''
for noticia in noticias:
    print(" ========== ========== ========= ======== =========")
    print("------------ ###### TITULO ###### ---------")
    print(noticia["Título"])
    print("/n")
    print("------------ ###### SUBTITULO ###### ---------")
    print(noticia["Subtítulo"])
    print("/n")
    print("------------ ###### CONTENIDO ###### ---------")
    print(noticia["Contenido"])
    print("/n")'''

'\nfor noticia in noticias:\n    print(" ========== ========== ========= ======== =========")\n    print("------------ ###### TITULO ###### ---------")\n    print(noticia["Título"])\n    print("/n")\n    print("------------ ###### SUBTITULO ###### ---------")\n    print(noticia["Subtítulo"])\n    print("/n")\n    print("------------ ###### CONTENIDO ###### ---------")\n    print(noticia["Contenido"])\n    print("/n")'

In [8]:
# GUARDADO DE LAS NOTICIAS EN UN JSON
import json
import os

def guardar_noticias(noticias, archivo_json):
    
    # Si el archivo existe, cargar su contenido
    if os.path.exists(archivo_json):
        with open(archivo_json, "r", encoding="utf-8") as f:
            datos_existentes = json.load(f)
    else:
        datos_existentes = []

    # Añadir las nuevas noticias
    datos_existentes.extend(noticias)

    # Guardar todo de nuevo
    with open(archivo_json, "w", encoding="utf-8") as f:
        json.dump(datos_existentes, f, ensure_ascii=False, indent=4)

# guardar_noticias(noticias=noticias)


In [ ]:
ruta_guardado = "./../../data/mediterraneodigital.json"

# ------------------------------------------------------------------------------------------------------------
# INTERNACIONAL
# ------------------------------------------------------------------------------------------------------------
print("Primera sección")
noticias = scrappeo_seccion(periodico="www.mediterraneodigital.com", nombre_periodico="Mediterráneo Digital",
                            seccion="internacional", nombre_seccion="Internacional",
                            extension="internacional", 
                            limite_peticiones=10, limite_urls=50)

guardar_noticias(noticias=noticias, archivo_json="../data/mediterraneodigital.json")

# ------------------------------------------------------------------------------------------------------------
# NACIONAL: ANDALUCÍA
# ------------------------------------------------------------------------------------------------------------
print("Segunda sección")
noticias = scrappeo_seccion(periodico="www.mediterraneodigital.com", nombre_periodico="Mediterráneo Digital",
                            seccion="espana/andalucia", nombre_seccion="Nacional",
                            extension="espana/andalucia", 
                            limite_peticiones=10, limite_urls=50)

guardar_noticias(noticias=noticias, archivo_json="../data/mediterraneodigital.json")

# ------------------------------------------------------------------------------------------------------------
# NACIONAL: CEUTA Y MELILLA
# ------------------------------------------------------------------------------------------------------------
print("Tercera sección")
noticias = scrappeo_seccion(periodico="www.mediterraneodigital.com", nombre_periodico="Mediterráneo Digital",
                            seccion="espana/ceuta-y-melilla", nombre_seccion="Nacional",
                            extension="espana/ceuta-y-melilla", 
                            limite_peticiones=10, limite_urls=50)

guardar_noticias(noticias=noticias, archivo_json="../data/mediterraneodigital.json")

# ------------------------------------------------------------------------------------------------------------
# SUCESOS
# ------------------------------------------------------------------------------------------------------------
print("Cuarta sección")
noticias = scrappeo_seccion(periodico="www.mediterraneodigital.com", nombre_periodico="Mediterráneo Digital",
                            seccion="sucesos-espana", nombre_seccion="Sucesos",
                            extension="sucesos-espana", 
                            limite_peticiones=10, limite_urls=50)

guardar_noticias(noticias=noticias, archivo_json="../data/mediterraneodigital.json")



Primera sección
Noticia 1 de 56
Noticia 2 de 56
Noticia 3 de 56
Noticia 4 de 56
Noticia 5 de 56
Noticia 6 de 56
Noticia 7 de 56
Noticia 8 de 56
Noticia 9 de 56
Noticia 10 de 56
Noticia 11 de 56
Noticia 12 de 56
Noticia 13 de 56
Noticia 14 de 56
Noticia 15 de 56
Noticia 16 de 56
Noticia 17 de 56
Noticia 18 de 56
Noticia 19 de 56
Noticia 20 de 56
Noticia 21 de 56
Noticia 22 de 56
Noticia 23 de 56
Noticia 24 de 56
Noticia 25 de 56
Noticia 26 de 56
Noticia 27 de 56
Noticia 28 de 56
Noticia 29 de 56
Noticia 30 de 56
Noticia 31 de 56
Noticia 32 de 56
Noticia 33 de 56
Noticia 34 de 56
Noticia 35 de 56
Noticia 36 de 56
Noticia 37 de 56
Noticia 38 de 56
Noticia 39 de 56
Noticia 40 de 56
Noticia 41 de 56
Noticia 42 de 56
Noticia 43 de 56
Noticia 44 de 56
Noticia 45 de 56
Noticia 46 de 56
Noticia 47 de 56
Noticia 48 de 56
Noticia 49 de 56
Noticia 50 de 56
Noticia 51 de 56
Noticia 52 de 56
Noticia 53 de 56
Noticia 54 de 56
Noticia 55 de 56
Noticia 56 de 56
Segunda sección
Noticia 1 de 57
Noticia 2

In [ ]:
import pandas as pd
# Debido ciertos fallos en la programación de las funciones anteriores, los datos guardados en el JSON, no son correctos
# este es un chunck extra donde solventamos esos pequeños problemas para no ejecutar de nuevo todo el código
# 1. Importar el JSON a un DataFrame
ruta_guardado = "./../../data/mediterraneodigital.json"
df = pd.read_json(ruta_guardado, orient="records")

# 2. Reemplazar una palabra en la columna 'categoría' mediante un condicional
# Ejemplo: si categoría contiene "política", cambiarla por "Politica Nacional"
df['Categoría'] = df['Categoría'].apply(
    lambda x: "Nacional" if "Sucesos" == x else x)

# 3. Eliminar registros donde el campo 'contenido' está vacío
df = df[df['Contenido'].str.strip() != ""]

# (Opcional) Reiniciar índice tras limpiar
df = df.reset_index(drop=True)


df.to_json(ruta_guardado, orient="records", force_ascii=False, indent=4)